In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("../Data/Smart_Farming_Crop_Yield_2024.csv")

df.head()

,farm_id,region,crop_type,soil_moisture_%,soil_pH,temperature_C,rainfall_mm,humidity_%,sunlight_hours,irrigation_type,...,sowing_date,harvest_date,total_days,yield_kg_per_hectare,sensor_id,timestamp,latitude,longitude,NDVI_index,crop_disease_status
0,FARM0001,North India,Wheat,35.95,5.99,17.79,75.62,77.03,7.27,NaN,...,2024-01-08,2024-05-09,122,4408.07,SENS0001,2024-03-19,14.970941,82.997689,0.63,Mild
1,FARM0002,South USA,Soybean,19.74,7.24,30.18,89.91,61.13,5.67,Sprinkler,...,2024-02-04,2024-05-26,112,5389.98,SENS0002,2024-04-21,16.613022,70.869009,0.58,NaN
2,FARM0003,South USA,Wheat,29.32,7.16,27.37,265.43,68.87,8.23,Drip,...,2024-02-03,2024-06-26,144,2931.16,SENS0003,2024-02-28,19.503156,79.068206,0.80,Mild
3,FARM0004,Central USA,Maize,17.33,6.03,33.73,212.01,70.46,5.03,Sprinkler,...,2024-02-21,2024-07-04,134,4227.80,SENS0004,2024-05-14,31.071298,85.519998,0.44,NaN
4,FARM0005,Central USA,Cotton,19.37,5.92,33.86,269.09,55.73,7.93,NaN,...,2024-02-05,2024-05-20,105,4979.96,SENS0005,2024-04-13,16.568540,81.691720,0.84,Severe


In [3]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.info()

Rows: 500
Columns: 22
<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   farm_id               500 non-null    str    
 1   region                500 non-null    str    
 2   crop_type             500 non-null    str    
 3   soil_moisture_%       500 non-null    float64
 4   soil_pH               500 non-null    float64
 5   temperature_C         500 non-null    float64
 6   rainfall_mm           500 non-null    float64
 7   humidity_%            500 non-null    float64
 8   sunlight_hours        500 non-null    float64
 9   irrigation_type       350 non-null    str    
 10  fertilizer_type       500 non-null    str    
 11  pesticide_usage_ml    500 non-null    float64
 12  sowing_date           500 non-null    str    
 13  harvest_date          500 non-null    str    
 14  total_days            500 non-null    int64  
 15  yield_kg_per

In [4]:
df.isnull().sum()

farm_id                   0
region                    0
crop_type                 0
soil_moisture_%           0
soil_pH                   0
temperature_C             0
rainfall_mm               0
humidity_%                0
sunlight_hours            0
irrigation_type         150
fertilizer_type           0
pesticide_usage_ml        0
sowing_date               0
harvest_date              0
total_days                0
yield_kg_per_hectare      0
sensor_id                 0
timestamp                 0
latitude                  0
longitude                 0
NDVI_index                0
crop_disease_status     130
dtype: int64

In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
print(df["region"].value_counts())
print(df["crop_type"].value_counts())
print(df["irrigation_type"].value_counts(dropna=False))
print(df["crop_disease_status"].value_counts(dropna=False))

region
Central USA    109
East Africa    107
North India     99
South USA       94
South India     91
Name: count, dtype: int64
crop_type
Maize      111
Soybean    108
Cotton     107
Wheat       92
Rice        82
Name: count, dtype: int64
irrigation_type
NaN          150
Sprinkler    121
Manual       118
Drip         111
Name: count, dtype: int64
crop_disease_status
Severe      133
NaN         130
Mild        125
Moderate    112
Name: count, dtype: int64


In [7]:
# Check missing values
missing_values = df.isnull().sum()

print("Missing values before cleaning:")
print(missing_values[missing_values > 0])

Missing values before cleaning:
irrigation_type        150
crop_disease_status    130
dtype: int64


In [8]:
# Fill missing categorical values with "Unknown"
df["irrigation_type"] = df["irrigation_type"].fillna("Unknown")
df["crop_disease_status"] = df["crop_disease_status"].fillna("Unknown")

# Check missing values again
print("Missing values after cleaning:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Missing values after cleaning:
Series([], dtype: int64)


In [9]:
# Check for duplicate records

duplicates = df.duplicated().sum()

print("Duplicate records:", duplicates)

Duplicate records: 0


In [10]:
# Remove duplicate records if any exist

df = df.drop_duplicates()

print("Dataset shape after removing duplicates:", df.shape)

Dataset shape after removing duplicates: (500, 22)


In [11]:
# Convert date columns to proper datetime format

df["sowing_date"] = pd.to_datetime(df["sowing_date"])
df["harvest_date"] = pd.to_datetime(df["harvest_date"])
df["timestamp"] = pd.to_datetime(df["timestamp"])

print(df[["sowing_date", "harvest_date", "timestamp"]].dtypes)

sowing_date     datetime64[us]
harvest_date    datetime64[us]
timestamp       datetime64[us]
dtype: object


In [12]:
# Calculate growing period from sowing and harvest dates

df["calculated_days"] = (
    df["harvest_date"] - df["sowing_date"]
).dt.days

# Compare with the existing total_days column

df["days_difference"] = (
    df["calculated_days"] - df["total_days"]
)

print(df[[
    "sowing_date",
    "harvest_date",
    "total_days",
    "calculated_days",
    "days_difference"
]].head(10))

  sowing_date harvest_date  total_days  calculated_days  days_difference
0  2024-01-08   2024-05-09         122              122                0
1  2024-02-04   2024-05-26         112              112                0
2  2024-02-03   2024-06-26         144              144                0
3  2024-02-21   2024-07-04         134              134                0
4  2024-02-05   2024-05-20         105              105                0
5  2024-01-13   2024-05-06         114              114                0
6  2024-03-04   2024-07-27         145              145                0
7  2024-01-24   2024-05-24         121              121                0
8  2024-03-12   2024-07-08         118              118                0
9  2024-01-18   2024-04-25          98               98                0


In [13]:
# Check whether the existing total_days values are correct

print(
    "Rows with different values:",
    (df["days_difference"] != 0).sum()
)

Rows with different values: 0


In [14]:
# Remove temporary columns used for verification

df.drop(columns=["calculated_days", "days_difference"], inplace=True)

print("Final columns:", df.columns.tolist())
print("Dataset shape:", df.shape)

Final columns: ['farm_id', 'region', 'crop_type', 'soil_moisture_%', 'soil_pH', 'temperature_C', 'rainfall_mm', 'humidity_%', 'sunlight_hours', 'irrigation_type', 'fertilizer_type', 'pesticide_usage_ml', 'sowing_date', 'harvest_date', 'total_days', 'yield_kg_per_hectare', 'sensor_id', 'timestamp', 'latitude', 'longitude', 'NDVI_index', 'crop_disease_status']
Dataset shape: (500, 22)


In [15]:
# Summary statistics for numerical columns

df.describe().T

,count,mean,min,25%,50%,75%,max,std
soil_moisture_%,500.0,26.75014,10.16,17.89,25.855,36.0225,44.98,10.150053
soil_pH,500.0,6.52398,5.51,6.03,6.53,7.04,7.5,0.585558
temperature_C,500.0,24.67574,15.0,20.295,24.655,29.09,34.84,5.348899
rainfall_mm,500.0,181.68574,50.17,119.2175,191.545,239.035,298.96,72.293091
humidity_%,500.0,65.19446,40.23,51.865,65.685,77.995,90.0,14.642849
sunlight_hours,500.0,7.03014,4.01,5.6675,6.995,8.47,10.0,1.69167
pesticide_usage_ml,500.0,26.58698,5.05,14.945,25.98,38.005,49.94,13.202429
sowing_date,500,2024-02-15 05:42:43.200000,2024-01-01 00:00:00,2024-01-23 00:00:00,2024-02-16 00:00:00,2024-03-09 00:00:00,2024-03-28 00:00:00,NaN
harvest_date,500,2024-06-13 17:36:57.600000,2024-04-09 00:00:00,2024-05-24 00:00:00,2024-06-14 00:00:00,2024-07-06 06:00:00,2024-08-17 00:00:00,NaN
total_days,500.0,119.496,90.0,105.75,119.0,134.0,150.0,16.798046


In [16]:
# Crop-wise performance

crop_performance = (
    df.groupby("crop_type")
      .agg(
          Average_Yield=("yield_kg_per_hectare", "mean"),
          Total_Yield=("yield_kg_per_hectare", "sum"),
          Number_of_Farms=("farm_id", "count")
      )
      .sort_values("Average_Yield", ascending=False)
)

crop_performance

,Average_Yield,Total_Yield,Number_of_Farms
crop_type,,,
Soybean,4256.814074,459735.92,108
Wheat,4077.584565,375137.78,92
Maize,3982.553874,442063.48,111
Cotton,3925.603084,420039.53,107
Rice,3896.180000,319486.76,82


In [17]:
# Region-wise performance

region_performance = (
    df.groupby("region")
      .agg(
          Average_Yield=("yield_kg_per_hectare", "mean"),
          Total_Yield=("yield_kg_per_hectare", "sum"),
          Number_of_Farms=("farm_id", "count")
      )
      .sort_values("Average_Yield", ascending=False)
)

region_performance

,Average_Yield,Total_Yield,Number_of_Farms
region,,,
South India,4122.884615,375182.50,91
East Africa,4053.184486,433690.74,107
Central USA,4013.083486,437426.10,109
North India,3996.221616,395625.94,99
South USA,3984.448830,374538.19,94


In [18]:
df.to_csv("smart_agriculture_cleaned.csv", index=False)